### **Representaciones de texto**
 
El objetivo del cuaderno es preparar al estudiante para temas de **representación textual**, **semántica vectorial** y el paso hacia
**representaciones multimodales**.

**Al finalizar este cuaderno, el estudiante debería poder:**
- distinguir representaciones discretas y distribuidas,
- construir representaciones **one-hot**, **BoW**, **n-gramas** y **TF-IDF**,
- entender el papel de las **subpalabras**,
- reconocer el paso hacia representaciones de **oraciones** y **documentos**.


#### **1. Del texto simbólico al espacio vectorial**

Los modelos clásicos de NLP requieren transformar el texto en vectores numéricos. 
Las primeras aproximaciones usan representaciones **discretas**: cada palabra ocupa una posición fija en el vocabulario.
Estas representaciones son simples y útiles para introducir la idea de espacio vectorial, pero sufren problemas de
**alta dimensionalidad**, **dispersión** y **falta de similitud semántica**.

En este cuaderno revisaremos la progresión:

1. **One-hot/label encoding**  
2. **Bag of Words (BoW)**  
3. **Bag of n-grams**  
4. **TF-IDF**  
5. **Subpalabras**  
6. **Panorama de representaciones de oraciones y documentos**


#### **2. One-hot encoding**

En **one-hot encoding**, cada palabra del vocabulario se representa por un vector de ceros con un único `1`.
Si el vocabulario tiene tamaño $V$, cada palabra se representa en $\mathbb{R}^V$.

Ventajas:
- muy simple,
- permite mapear texto a vectores de manera explícita.

Limitaciones:
- vectores muy dispersos,
- no modela similitud semántica,
- escala mal cuando el vocabulario crece.


In [ ]:
documentos = ["Dog bites man.", "Man bites dog.", "Dog eats meat.", "Man eats food."]
docs_procesados = [doc.lower().replace(".", "") for doc in documentos]
print(docs_procesados)

Aquí se obtiene una lista de documentos en la que cada texto está en minúsculas y sin puntos. Esto facilita la construcción de un vocabulario sin duplicidades causadas por diferencias de mayúsculas/minúsculas o puntuación.

**Construcción del vocabulario y codificación One-Hot**

El primer paso es construir el vocabulario, es decir, asignar a cada palabra única un identificador numérico. Se recorre cada documento y se separa en palabras:

In [ ]:
vocab = {}
for doc in docs_procesados:
    for palabra in doc.split():
        if palabra not in vocab:
            vocab[palabra] = len(vocab)
print(vocab)

In [ ]:
def obtiene_vector_onehot(cadena):
    onehot_codificado = []
    for palabra in cadena.split():
        temp = [0] * len(vocab)
        if palabra in vocab:
            temp[vocab[palabra]] = 1
        onehot_codificado.append(temp)
    return onehot_codificado

print("One-hot para 'dog bites man':")
for v in obtiene_vector_onehot("dog bites man"):
    print(v)

##### **2.1 Codificación con `scikit-learn`**

En la práctica, suele usarse una librería para evitar errores de implementación y mantener compatibilidad con pipelines de preprocesamiento.


##### **Codificación one-hot y label encoding con scikit-learn**

La librería **scikit-learn** ofrece herramientas optimizadas para realizar tanto la codificación one-hot como la codificación de etiquetas (label encoding). Estas técnicas se utilizan para transformar las palabras en representaciones numéricas.

**Label encoding**

La **codificación de etiquetas** convierte cada palabra en un valor numérico único que va de `0` a `n-1`, donde `n` es el número de palabras únicas en el corpus. El siguiente fragmento de código muestra cómo hacerlo:

In [ ]:
from sklearn.preprocessing import LabelEncoder

S1 = 'dog bites man'
S2 = 'man bites dog'
S3 = 'dog eats meat'
S4 = 'man eats food'

data = [S1.split(), S2.split(), S3.split(), S4.split()]
# Se concatenan todas las palabras de cada documento en una única lista
valores = data[0] + data[1] + data[2] + data[3]
print("Los datos: ", valores)

# Instancia y ajusta el LabelEncoder
le = LabelEncoder()
le.fit(valores)
# Transforma una lista de palabras usando el encoder
etiquetas = le.transform(valores)
print("Codificación de etiquetas: ", etiquetas)

**One-Hot encoding con scikit-learn**

Para realizar la codificación one-hot se utiliza el objeto `OneHotEncoder` de scikit-learn, el cual transforma los valores etiquetados en una matriz dispersa (sparse matrix) en la que cada columna representa una de las posibles categorías:

In [ ]:
from sklearn.preprocessing import OneHotEncoder
import numpy as np

valores = ['rojo', 'verde', 'azul', 'verde', 'rojo']
valores = np.array(valores).reshape(-1, 1)

ohe = OneHotEncoder(sparse_output=False) 
ohe.fit(valores)

# Transforma los datos
datos_onehot = ohe.transform(valores)
print("Codificación One-Hot:\n", datos_onehot)


Con estos ejemplos se comprende cómo se asigna a cada palabra una representación numérica, tanto en forma de etiqueta única como en un vector binario one-hot.

#### **3. Bolsa de palabras (Bag of Words, BoW)**

En **BoW**, un documento se representa por el conteo de las palabras que contiene, sin considerar el orden.
Esta representación sigue siendo discreta, pero ya permite trabajar a nivel de documento.

Idea:
- construir un vocabulario;
- contar cuántas veces aparece cada término en cada texto;
- representar cada documento como un vector de frecuencias.


A continuación, realizaremos la tarea de encontrar la representación de una bolsa de palabras. Utilizaremos [CountVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html) de sklearn.


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

corpus = [
    "el gato duerme",
    "el perro duerme",
    "el gato y el perro juegan"
]


vectorizer = CountVectorizer()
X = vectorizer.fit_transform(corpus)

print("Vocabulario:", vectorizer.get_feature_names_out())
print("Matriz BoW:\n", X.toarray())


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# Lista de documentos
docs_procesados = ["dog bites man", "man bites dog", "dog and dog are friends"]
print("El corpus: ", docs_procesados)

# Inicializa el vectorizador
count_vect = CountVectorizer()

# Construcción de la representación BoW para el corpus
bow_rep = count_vect.fit_transform(docs_procesados)  # Aquí se ajusta y transforma

# Mapeo del vocabulario
print("El vocabulario: ", count_vect.vocabulary_)

# Ve la representación BoW para los dos primeros documentos
print("Representación BoW para 'dog bites man': ", bow_rep[0].toarray())
print("Representación BoW para 'man bites dog': ", bow_rep[1].toarray())

# Representación usando este vocabulario para un nuevo texto
temp = count_vect.transform(["dog and dog are friends"])
print("Representación BoW para 'dog and dog are friends':", temp.toarray())


#### **Explicación**

```python
['dog bites man', 'man bites dog', 'dog and dog are friends']
```

Este es tu conjunto de 3 documentos (frases).

**Vocabulario generado**

```python
{'dog': 3, 'bites': 2, 'man': 5, 'and': 0, 'are': 1, 'friends': 4}
```

El `CountVectorizer` asignó un **ID único** (índice de columna en el vector BoW) a cada palabra que aparece en el corpus completo.

| Palabra   | Índice |
|-----------|--------|
| 'and'     | 0      |
| 'are'     | 1      |
| 'bites'   | 2      |
| 'dog'     | 3      |
| 'friends' | 4      |
| 'man'     | 5      |

Entonces cada documento se representa como un vector de 6 elementos (uno por cada palabra).

**Representación BoW para cada documento**

**Documento 1:** `'dog bites man'`
Vector BoW: `[[0 0 1 1 0 1]]`

| Palabra   | Conteo |
|-----------|--------|
| 'and'     | 0      |
| 'are'     | 0      |
| 'bites'   | 1      |
| 'dog'     | 1      |
| 'friends' | 0      |
| 'man'     | 1      |

El vector indica que las palabras `'dog'`, `'bites'` y `'man'` aparecen una vez en ese documento.

**Documento 2:** `'man bites dog'`

Vector BoW : `[[0 0 1 1 0 1]]`

Es **idéntico al documento 1**, ya que tiene las mismas palabras, solo que en otro orden (que BoW **ignora**).

**Documento 3:** `'dog and dog are friends'`

Vector BoW: `[[1 1 0 2 1 0]]`

| Palabra   | Conteo |
|-----------|--------|
| 'and'     | 1      |
| 'are'     | 1      |
| 'bites'   | 0      |
| 'dog'     | 2      |
| 'friends' | 1      |
| 'man'     | 0      |

Aquí se ve que `'dog'` aparece **dos veces**, y que `'and'`, `'are'`, y `'friends'` aparecen **una vez**.

**Resultados**

- Cada vector tiene 6 posiciones, una por palabra en el vocabulario.
- La posición `i` representa el número de veces que la palabra con índice `i` aparece en ese documento.
- El modelo **ignora el orden** de las palabras y se enfoca solo en **la frecuencia de aparición**.


##### **Aplicación a nuevos textos y uso de vectores binarios**

Una ventaja del método BoW es que, una vez construido el vocabulario, se pueden transformar nuevos textos utilizando el mismo esquema. Por ejemplo:


In [ ]:
# Configura CountVectorizer para vectores binarios
count_vect_bin = CountVectorizer(binary=True)
bow_rep_bin = count_vect_bin.fit_transform(docs_procesados)

temp_bin = count_vect_bin.transform(["dog and dog are friends"])
print("Representacion Bow (binaria) para 'dog and dog are friends':", temp_bin.toarray())

Esto da como resultado una representación diferente para la misma oración. `CountVectorizer` admite n-gramas tanto de palabras como de caracteres. 

#### **4. Bolsa de n-gramas**

El problema principal de BoW es que ignora completamente el orden.
Una forma simple de recuperar algo de estructura es usar **n-gramas**, es decir, secuencias contiguas de `n` tokens.

- unigramas: palabras individuales;
- bigramas: pares consecutivos;
- trigramas: ternas consecutivas.

Esto permite capturar fragmentos como “machine learning” o “deep learning”.


#### **Bolsa de n-gramas**

Los esquemas de representación que hemos visto hasta ahora tratan las palabras como unidades independientes, sin tener en cuenta las frases ni el orden en que aparecen. El enfoque de la **bolsa de n-gramas** (BoN) intenta remediar esta limitación dividiendo el texto en fragmentos de `n` palabras (o *tokens*) contiguas. Esto permite capturar algo de contexto, lo cual no era posible con los enfoques anteriores. Cada uno de estos fragmentos se denomina *n-grama*.

El vocabulario del corpus, `V`, se define como la colección de todos los *n-gramas* únicos presentes en el texto. Luego, cada documento del corpus se representa mediante un vector de longitud `|V|`, donde cada componente del vector contiene el recuento de frecuencia de los *n-gramas* presentes en el documento, asignando un valor de cero a los *n-gramas* que no aparecen.

En el ámbito del procesamiento del lenguaje natural (*NLP*), este esquema también se conoce como **selección de características basada en n-gramas**.


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# Corpus de ejemplo
docs_procesados = ['dog bites man', 'man bites dog', 'dog and dog are friends']
print("El corpus: ", docs_procesados)

# Vectorización con bigramas
count_vect = CountVectorizer(ngram_range=(2, 2))

# Construcción de la representación BoW para el corpus
bow_rep = count_vect.fit_transform(docs_procesados)

# Mapeo del vocabulario
print("El vocabulario: ", count_vect.vocabulary_)

# Representación BoW para los dos primeros documentos
print("Representacion BoW para 'dog bites man': ", bow_rep[0].toarray())
print("Representacion BoW para 'man bites dog': ", bow_rep[1].toarray())

# Representación usando el mismo vocabulario para un nuevo texto
temp = count_vect.transform(["dog and dog are friends"])
print("Representación BoW para 'dog and dog are friends':", temp.toarray())


#### **5. TF-IDF**

**TF-IDF** pondera los términos por dos factores:

- **TF (term frequency):** cuánto aparece un término dentro del documento.
- **IDF (inverse document frequency):** qué tan raro o informativo es un término en el corpus.

La intuición es sencilla: palabras frecuentes en un documento son útiles, pero palabras demasiado frecuentes en todos los documentos
suelen aportar poca información.


#### **TF-IDF**

El método **TF-IDF** es una técnica que pondera la frecuencia de las palabras en un documento en relación a su frecuencia en el corpus completo. Este método ayuda a disminuir el peso de términos muy frecuentes (que no discriminan bien entre documentos) y aumentar el de aquellos que son raros y más informativos.

**Cálculo de TF-IDF**

- **TF (frecuencia de término):** Mide cuántas veces aparece una palabra en un documento. Se puede normalizar para evitar favorecer documentos más largos.
- **IDF (frecuencia inversa de documento):** Calcula la importancia de una palabra a partir de la cantidad de documentos en los que aparece. Se formula tomando el logaritmo de la razón entre el número total de documentos y la cantidad de documentos que contienen la palabra.
- **Producto TF-IDF:** Multiplicar TF por IDF da el valor final que indica la relevancia de una palabra en el contexto del documento y el corpus.


$$
TF\text{-}IDF(t,d) = TF(t,d) \times IDF(t)
$$

Donde `t` es el término, `d` es el documento, y el corpus es el conjunto total de documentos.

Scikit-learn proporciona el objeto `TfidfVectorizer` para calcular esta representación:


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Instancia TfidfVectorizer
tfidf = TfidfVectorizer()
bow_rep_tfidf = tfidf.fit_transform(docs_procesados)

# Imprime los valores IDF para cada palabra en el vocabulario
print("IDF para todas las palabras en el vocabulario:", tfidf.idf_)
print("-" * 10)
print("Todas las palabras en el vocabulario:", tfidf.get_feature_names_out())
print("-" * 10)

# Muestra la representación TF-IDF para todo el corpus
print("Representacion TFIDF para todos los documentos en el corpus:\n", bow_rep_tfidf.toarray())
print("-" * 10)

# Transformar un nuevo texto usando el mismo modelo TF-IDF
temp = tfidf.transform(["dog and man are friends"])
print("Representacion Tfidf para 'dog and man are friends':\n", temp.toarray())

#### **Explicación de resultados**

**Corpus original**

```python
docs_procesados = [
    'dog bites man', 
    'man bites dog', 
    'dog and dog are friends'
]
```

**Paso 1: Palabras en el vocabulario**

```python
['and', 'are', 'bites', 'dog', 'friends', 'man']
```

Esto es el vocabulario ordenado que el `TfidfVectorizer` extrajo. Cada posición del vector representa una de estas palabras.

**Paso 2: Valores IDF**

```python
[1.6931, 1.6931, 1.2877, 1.0000, 1.6931, 1.2877]
```

Esto nos dice qué tan **rara** es cada palabra en el corpus:

| Palabra   | IDF     | Significado                                      |
|-----------|---------|--------------------------------------------------|
| 'dog'     | 1.000   | Aparece en **todos** los documentos -> menos útil |
| 'bites'   | 1.2877  | Aparece en 2/3 documentos                        |
| 'man'     | 1.2877  | Aparece en 2/3 documentos                        |
| 'and'     | 1.6931  | Solo aparece en 1 documento -> más informativa   |
| 'are'     | 1.6931  | Solo aparece en 1 documento -> más informativa   |
| 'friends' | 1.6931  | Solo aparece en 1 documento -> más informativa   |

Cuanto **mayor el IDF**, más valiosa es la palabra como discriminante.

**Paso 3: Representación TF-IDF del corpus**

```python
[[0.         0.         0.6198  0.4813  0.         0.6198 ]
 [0.         0.         0.6198  0.4813  0.         0.6198 ]
 [0.4770     0.4770     0.       0.5634  0.4770     0.      ]]
```

Cada fila es un documento, y cada columna es una palabra del vocabulario:

Documento 1:`'dog bites man'`

| Palabra   | TF-IDF |
|-----------|--------|
| 'dog'     | 0.4813 |
| 'bites'   | 0.6198 |
| 'man'     | 0.6198 |

Las otras palabras no aparecen, por eso su valor es 0.

##### Documento 3: `'dog and dog are friends'`

| Palabra   | TF-IDF                             |
|-----------|-------------------------------------|
| `dog`     | 0.5634 (ocurre dos veces, pero es común) |
| `and`     | 0.4770                              |
| `are`     | 0.4770                              |
| `friends` | 0.4770                              |


##### Paso 4: TF-IDF para nuevo texto:  
`'dog and man are friends'`

```python
[[0.5046  0.5046  0.      0.2980  0.5046  0.3838]]
```

| Palabra   | TF-IDF                                 |
|-----------|-----------------------------------------|
| `dog`     | 0.2980 -> común en el corpus             |
| `and`     | 0.5046 -> rara en el corpus              |
| `are`     | 0.5046 -> rara en el corpus              |
| `friends` | 0.5046 -> rara en el corpus              |
| `man`     | 0.3838 -> aparece moderadamente          |


Esto refleja que `'dog'` tiene **menos peso** que `'friends'`, `'are'`, `'and'`, etc., ya que aparece **en casi todos los documentos**, mientras que las otras solo aparecen en uno -> son más informativas.


##### **Visualización en Pandas**

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# Corpus original
docs_procesados = ['dog bites man', 'man bites dog', 'dog and dog are friends']

# Instancia y ajusta el vectorizador
tfidf = TfidfVectorizer()
bow_rep_tfidf = tfidf.fit_transform(docs_procesados)

# Obtiene las palabras del vocabulario
palabras = tfidf.get_feature_names_out()

# Crea DataFrame para visualizar el corpus representado como TF-IDF
df_tfidf = pd.DataFrame(bow_rep_tfidf.toarray(), columns=palabras, index=[f'Doc_{i+1}' for i in range(len(docs_procesados))])
print("Representación TF-IDF del corpus:")
print(df_tfidf)
print("-" * 40)

# Transforma un nuevo texto
nuevo_texto = "dog and man are friends"
temp = tfidf.transform([nuevo_texto])

# Crea DataFrame para el nuevo texto
df_nuevo = pd.DataFrame(temp.toarray(), columns=palabras, index=['Nuevo texto'])
print("Representación TF-IDF para el nuevo texto:")
print(df_nuevo)


#### **6. Subpalabras**

Las representaciones basadas en palabras completas sufren cuando aparecen:
- palabras raras,
- variaciones morfológicas,
- palabras fuera de vocabulario.

Una solución intermedia es trabajar con **subpalabras** o fragmentos internos.
Esto resulta especialmente útil en idiomas con mucha flexión o en dominios técnicos.

Ejemplos:
- `computación` → `comput`, `ación`
- `multimodalidad` → `multi`, `modal`, `idad`

Los modelos modernos suelen combinar la noción de token y subpalabra para mejorar cobertura léxica.


##### **6.1 Ejemplo simple de subpalabras con n-gramas de caracteres**

Sin usar bibliotecas externas, podemos aproximar la idea de subpalabras mediante n-gramas de caracteres.
No reemplaza a métodos como SentencePiece o BPE, pero sirve para visualizar el concepto.


In [ ]:
def char_ngrams(word, n=3):
    word = f"<{word}>"
    return [word[i:i+n] for i in range(len(word)-n+1)]

palabras = ["computacion", "multimodalidad", "aprendizaje"]
for p in palabras:
    print(p, "->", char_ngrams(p, 3))

#### **7. Representaciones de oraciones y documentos**

Cuando el objetivo deja de ser representar palabras individuales y pasa a ser representar unidades más grandes, aparecen nuevas estrategias:

- **Promedio de embeddings**: sumar o promediar vectores de palabras.
- **Doc2Vec**: aprender representaciones densas de documentos.
- **Embeddings de oraciones**: obtener un vector para toda la oración.
- **Modelos contextuales**: representar un token según su contexto.

Para un repaso, lo importante no es dominar todos estos métodos, sino entender que existe una transición:

**palabra → frase → oración → documento**

y que cada nivel requiere mecanismos de composición o aprendizaje más ricos.


##### **7.1 ¿Por qué esto importa para multimodalidad?**

En aprendizaje multimodal temprano, muchas veces se necesita alinear:
- una **palabra** con una imagen,
- una **oración** con una imagen,
- o una **descripción completa** con contenido visual.

Por eso este recorrido es importante: antes de alinear texto con imagen, primero hay que decidir
**qué representación textual** se va a usar.


#### **8. Ejercicios de repaso**
**1. Explica con tus palabras la diferencia entre one-hot y Bag of Words.**

La representación one-hot codifica cada palabra mediante un vector donde una única posición vale 1 y las demás 0. Por ejemplo, la palabra "gol" tendría un vector distinto al de "penal". En cambio, Bag of Words representa un documento completo contando cuántas veces aparecen palabras como "gol", "partido", "arquero" o "campeonato" dentro de una descripción deportiva.

**2. ¿Qué problema intenta resolver TF-IDF respecto de BoW?**

TF-IDF reduce la importancia de palabras muy frecuentes y resalta aquellas más informativas. Por ejemplo, en noticias deportivas palabras como "equipo" o "partido" aparecen en casi todos los textos, mientras que términos como "tiro libre", "penal" o "hat-trick" pueden aportar más información para diferenciar eventos específicos.

**3. Da un ejemplo donde los n-gramas sean preferibles a unigramas.**

En deportes, expresiones como "tiro libre", "tarjeta roja" o "salto con garrocha" tienen un significado específico. Si se separan en unigramas, parte del significado se pierde. Los bigramas permiten conservar mejor estas expresiones compuestas.

**4. ¿Por qué las subpalabras ayudan con palabras raras o fuera de vocabulario?**

Las subpalabras permiten representar términos poco frecuentes o nuevos. Por ejemplo, si aparece el apellido de un deportista que no estaba en el vocabulario, el modelo puede dividirlo en fragmentos conocidos y construir una representación razonable sin haber visto previamente la palabra completa.

**5. ¿Qué ventajas tendría usar una representación de oración en lugar de una representación palabra por palabra?**

Una representación de oración captura el significado global de descripciones como "un jugador anota un gol de cabeza en los minutos finales". Esto resulta más útil que analizar cada palabra por separado, especialmente en sistemas multimodales como OpenCLIP, donde la descripción completa se compara con imágenes o videos deportivos mediante embeddings y similitud coseno.


In [ ]:
# Escribe aquí tus respuestas


#### **9. Cierre**

Este cuaderno resume el paso desde representaciones discretas simples hasta una visión más rica del texto:

- **one-hot** y **label encoding** para introducir el vocabulario,
- **BoW** y **n-gramas** para representar documentos,
- **TF-IDF** para ponderar términos informativos,
- **subpalabras** para mejorar cobertura,
- y un panorama de representaciones de **oraciones** y **documentos**.

Este marco prepara la transición hacia **embeddings distribuidos** y más adelante, hacia el problema de
**alineamiento texto-imagen** en aprendizaje multimodal.
